# MSR-VTT Video Retrieval — v2 (Optimized)

**Fixes vs v1:**
1. Train/eval consistency: bỏ KAN pair-fusion, dùng dual-encoder độc lập (video branch + text branch)
2. Thêm MedR metric, bỏ Rsum khỏi early stopping → dùng t2v_R@1
3. Mean-caption aggregation trong feature extraction
4. Tăng num_frames: 8→12, num_epochs: 50→60, patience: 10→15
5. Hard negative mining thực sự với in-batch negatives + margin
6. Evaluation đúng MSR-VTT protocol: 1K test set, mean caption per video
7. Checkpoint theo t2v_R@1 thay vì Rsum

In [ ]:
# CELL 1 — Install / Verify
import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

try:
    import einops
except ImportError:
    pip('einops')

import torch, torchvision, transformers, cv2
print(f'torch        : {torch.__version__}')
print(f'transformers : {transformers.__version__}')
print(f'CUDA         : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU          : {p.name}  ({p.total_memory/1e9:.1f} GB)')

In [ ]:
# CELL 2 — Imports & Reproducibility
import os, json, random, logging, time
from pathlib import Path
from dataclasses import dataclass
from typing import List, Optional, Dict
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
import cv2
from tqdm.notebook import tqdm
from transformers import CLIPModel, CLIPProcessor, get_cosine_schedule_with_warmup
import torch.backends.cudnn as cudnn

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger(__name__)
cudnn.benchmark = True

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print('Imports OK')

In [ ]:
# CELL 3 — Configuration (v2 — optimized)
@dataclass
class Config:
    # ── Paths ──────────────────────────────────────────────────────────
    BASE_DIR   : str = '/kaggle/input/msr-vtt'
    train_json : str = BASE_DIR + '/MSR-VTT/Video_Captions/Train_Captions.json'
    val_json   : str = BASE_DIR + '/MSR-VTT/Video_Captions/Val_Captions.json'
    test_json  : str = BASE_DIR + '/MSR-VTT/Video_Captions/Test_Captions.json'
    train_video_dir : str = BASE_DIR + '/MSR-VTT/Video_Datasets/Train'
    val_video_dir   : str = BASE_DIR + '/MSR-VTT/Video_Datasets/Val'
    test_video_dir  : str = BASE_DIR + '/MSR-VTT/Video_Datasets/Test'
    feat_dir   : str = '/kaggle/working/features_v2'
    ckpt_dir   : str = '/kaggle/working/checkpoints_v2'

    # ── CLIP backbone ──────────────────────────────────────────────────
    clip_model     : str   = 'openai/clip-vit-large-patch14'
    embed_dim      : int   = 1024   # ViT-L/14 vision pooler_output
    text_embed_dim : int   = 768    # ViT-L/14 text pooler_output
    proj_dim       : int   = 512    # [v2] giảm xuống 512 — tránh overfit, chuẩn CLIP

    # ── Video sampling ─────────────────────────────────────────────────
    num_frames  : int = 12          # [v2] tăng từ 8 → 12
    frame_size  : int = 224

    # ── Head architecture — Dual Encoder (KHÔNG KAN fusion) ────────────
    # [v2] KAN pair-fusion bị loại bỏ vì gây train/eval inconsistency.
    # Thay bằng lightweight MLP projection sau SE+MoE.
    sts_shift_ratio : float = 0.125
    num_experts     : int   = 4
    top_k_experts   : int   = 2
    moe_hidden_dim  : int   = 1024
    head_hidden_dim : int   = 512   # MLP hidden sau MoE

    # ── Training ───────────────────────────────────────────────────────
    batch_size      : int   = 128   # [v2] tăng từ 64 → 128 (hard negatives tốt hơn)
    accum_steps     : int   = 2     # [v2] giảm từ 4 → 2 (batch lớn hơn rồi)
    num_epochs      : int   = 60    # [v2] tăng từ 50 → 60
    lr_proj         : float = 5e-5  # [v2] LR nhỏ hơn cho proj (CLIP đã pretrained)
    lr_head         : float = 2e-4  # [v2] giảm từ 3e-4
    weight_decay    : float = 0.02
    warmup_ratio    : float = 0.1   # [v2] tăng warmup
    grad_clip       : float = 1.0
    temperature_init: float = 0.07
    logit_scale_max : float = 4.6052  # [v2] ln(100) — chuẩn CLIP
    label_smoothing : float = 0.1
    patience        : int   = 15    # [v2] tăng patience
    min_delta       : float = 0.3
    eval_every      : int   = 1
    use_amp         : bool  = False
    log_every       : int   = 20
    eval_batch_size : int   = 512

    # ── Feature extraction ─────────────────────────────────────────────
    extract_batch_size: int = 128
    num_workers       : int = 2

cfg = Config()
cfg.device = 'cuda' if torch.cuda.is_available() else 'cpu'

Path(cfg.feat_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.ckpt_dir).mkdir(parents=True, exist_ok=True)
print(f'Device         : {cfg.device}')
print(f'proj_dim       : {cfg.proj_dim}  [v2: 512]')
print(f'num_frames     : {cfg.num_frames}  [v2: 12]')
print(f'batch_size     : {cfg.batch_size}  [v2: 128]')

In [ ]:
# CELL 4 — MSR-VTT JSON Parser & Video Decoder

CLIP_MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073])
CLIP_STD  = torch.tensor([0.26862954, 0.26130258, 0.27577711])


def load_msrvtt_json(json_path: str, video_dir: str) -> List[Dict]:
    with open(json_path, 'r', encoding='utf-8') as f:
        raw = json.load(f)

    records = []
    video_dir = Path(video_dir)

    for key, entries in raw.items():
        if not entries:
            continue
        video_id = entries[0]['video_id']
        captions = [e['caption'] for e in entries if e.get('caption')]

        video_path = video_dir / f'{video_id}.mp4'
        if not video_path.exists():
            video_path = video_dir / f'{video_id}.avi'
        if not video_path.exists():
            logger.warning(f'Video not found: {video_id} in {video_dir}')
            continue

        records.append({
            'video_id': video_id,
            'video'   : str(video_path),
            'captions': captions,
            'source'  : 'MSR-VTT'
        })

    logger.info(f'Loaded {len(records)} videos from {json_path}')
    return records


def frames_to_tensor(frames: list, device) -> torch.Tensor:
    tensors = []
    for f in frames:
        f_rgb = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        h, w  = f_rgb.shape[:2]
        scale = max(224 / h, 224 / w)
        nh    = max(224, int(round(h * scale)))
        nw    = max(224, int(round(w * scale)))
        f_rsz = cv2.resize(f_rgb, (nw, nh), interpolation=cv2.INTER_LINEAR)
        top   = (nh - 224) // 2
        left  = (nw - 224) // 2
        f_crop = f_rsz[top:top+224, left:left+224]
        t = torch.from_numpy(f_crop).permute(2, 0, 1).float() / 255.0
        tensors.append(t)
    out = torch.stack(tensors, dim=0).to(device)
    mean = CLIP_MEAN.to(device).view(1, 3, 1, 1)
    std  = CLIP_STD.to(device).view(1, 3, 1, 1)
    return (out - mean) / std


def sample_frames(video_path: str, num_frames: int = 12) -> list:
    """Uniformly sample num_frames từ video."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, total - 1, num_frames, dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if ret:
            frames.append(frame)
    cap.release()
    while len(frames) < num_frames and frames:
        frames.append(frames[-1])
    return frames[:num_frames]


train_data = load_msrvtt_json(cfg.train_json, cfg.train_video_dir)
val_data   = load_msrvtt_json(cfg.val_json,   cfg.val_video_dir)
test_data  = load_msrvtt_json(cfg.test_json,  cfg.test_video_dir)

print(f'Train videos : {len(train_data)}')
print(f'Val   videos : {len(val_data)}')
print(f'Test  videos : {len(test_data)}')

In [ ]:
# CELL 5 — CLIP Backbone

class CLIPBackbone(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.model     = CLIPModel.from_pretrained(cfg.clip_model)
        self.processor = CLIPProcessor.from_pretrained(cfg.clip_model)
        self.cfg = cfg

        # [v2] proj_dim=512 (CLIP native)
        self.vision_proj = nn.Linear(cfg.embed_dim,      cfg.proj_dim, bias=False)
        self.text_proj   = nn.Linear(cfg.text_embed_dim, cfg.proj_dim, bias=False)

        # [v2] Khởi tạo proj gần identity-like để bảo toàn CLIP representation
        nn.init.xavier_uniform_(self.vision_proj.weight)
        nn.init.xavier_uniform_(self.text_proj.weight)

        for p in self.model.parameters():
            p.requires_grad_(False)

    def encode_frames(self, frames_tensor: torch.Tensor) -> torch.Tensor:
        """
        frames_tensor: (N_frames, 3, 224, 224)
        returns: (proj_dim,) — mean pooled & projected
        """
        out   = self.model.vision_model(pixel_values=frames_tensor)
        feats = out.pooler_output        # (N_frames, 1024)
        feats = feats.mean(dim=0)        # (1024,)
        return F.normalize(self.vision_proj(feats), dim=-1)

    def encode_text(self, captions: List[str]) -> torch.Tensor:
        """
        captions: list of strings
        returns: (len(captions), proj_dim)
        """
        inputs = self.processor(
            text=captions,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=77
        ).to(next(self.model.parameters()).device)
        out   = self.model.text_model(**{k: v for k, v in inputs.items()})
        feats = out.pooler_output        # (B, 768)
        return F.normalize(self.text_proj(feats), dim=-1)

    def unfreeze_last_n_layers(self, n: int = 2):
        layers = self.model.vision_model.encoder.layers
        for layer in layers[-n:]:
            for p in layer.parameters():
                p.requires_grad_(True)
        logger.info(f'Unfrozen last {n} vision encoder layers.')


print('CLIPBackbone defined.')

In [ ]:
# CELL 6 — Modules: STS, SE, MoE
# [v2] Loại bỏ KAN hoàn toàn — KAN pair-fusion gây train/eval inconsistency
# thay bằng MLP head nhẹ sau mỗi branch

class ShiftedTokenShift(nn.Module):
    def __init__(self, dim: int, shift_ratio: float = 0.125):
        super().__init__()
        self.shift = max(1, int(dim * shift_ratio))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = x.clone()
        out[:, :self.shift] = torch.roll(x[:, :self.shift], 1, dims=0)
        return out


class SEBlock(nn.Module):
    def __init__(self, dim: int, reduction: int = 4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(dim, dim // reduction, bias=False),
            nn.ReLU(),
            nn.Linear(dim // reduction, dim, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.fc(x)


class SparseMoE(nn.Module):
    def __init__(self, dim: int, num_experts: int, top_k: int, hidden_dim: int):
        super().__init__()
        self.top_k = top_k
        self.gate  = nn.Linear(dim, num_experts, bias=False)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(dim, hidden_dim, bias=False),
                nn.GELU(),
                nn.Linear(hidden_dim, dim, bias=False)
            ) for _ in range(num_experts)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate_logits = self.gate(x)
        top_k_vals, top_k_idx = gate_logits.topk(self.top_k, dim=-1)
        gate_w = F.softmax(top_k_vals, dim=-1)
        out = torch.zeros_like(x)
        for k in range(self.top_k):
            for e in range(len(self.experts)):
                mask = (top_k_idx[:, k] == e)
                if mask.any():
                    out[mask] += gate_w[mask, k:k+1] * self.experts[e](x[mask])
        return out


class ProjectionMLP(nn.Module):
    """[v2] Thay thế KAN: MLP projection nhanh hơn, consistent train/eval."""
    def __init__(self, dim: int, hidden_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim, bias=False),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, dim, bias=False),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x) + x  # residual


print('STS / SE / MoE / ProjectionMLP defined. [KAN removed — v2]')

In [ ]:
# CELL 7 — DualEncoderHead (v2)
# KEY FIX: video branch và text branch HOÀN TOÀN ĐỘC LẬP
# → evaluate() dùng CÙNG pipeline với training (không bypass gì cả)

class DualEncoderHead(nn.Module):
    """
    v2: Dual Encoder — video và text branch độc lập.
    KHÔNG có cross-modal fusion tại thời điểm encoding.
    Similarity được tính bằng cosine(v_out, t_out).

    Pipeline cho mỗi branch:
      input → STS → MoE → SE → ProjectionMLP → L2 normalize → output
    """
    def __init__(self, cfg):
        super().__init__()
        d = cfg.proj_dim
        self.sts   = ShiftedTokenShift(d, cfg.sts_shift_ratio)
        self.moe_v = SparseMoE(d, cfg.num_experts, cfg.top_k_experts, cfg.moe_hidden_dim)
        self.moe_t = SparseMoE(d, cfg.num_experts, cfg.top_k_experts, cfg.moe_hidden_dim)
        self.se_v  = SEBlock(d)
        self.se_t  = SEBlock(d)
        self.proj_v = ProjectionMLP(d, cfg.head_hidden_dim)
        self.proj_t = ProjectionMLP(d, cfg.head_hidden_dim)
        self.logit_scale = nn.Parameter(torch.tensor(np.log(1 / cfg.temperature_init)))

    def encode_video(self, vid_emb: torch.Tensor) -> torch.Tensor:
        """vid_emb: (B, D) → v_out: (B, D) normalized"""
        x = F.normalize(vid_emb, dim=-1)
        x = self.se_v(self.moe_v(self.sts(x)))
        x = self.proj_v(x)
        return F.normalize(x, dim=-1)

    def encode_text(self, txt_emb: torch.Tensor) -> torch.Tensor:
        """txt_emb: (B, D) → t_out: (B, D) normalized"""
        x = F.normalize(txt_emb, dim=-1)
        x = self.se_t(self.moe_t(self.sts(x)))
        x = self.proj_t(x)
        return F.normalize(x, dim=-1)

    def forward(self, vid_emb: torch.Tensor, txt_emb: torch.Tensor) -> tuple:
        v_out = self.encode_video(vid_emb)
        t_out = self.encode_text(txt_emb)
        scale = self.logit_scale.exp().clamp(max=100.0)
        return v_out, t_out, scale


print('DualEncoderHead defined. [v2 — KAN-free, consistent train/eval]')

In [ ]:
# CELL 8 — Loss (giữ nguyên ContrastiveLoss, tương thích DualEncoderHead)

class ContrastiveLoss(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.margin = cfg.hard_neg_margin if hasattr(cfg, 'hard_neg_margin') else 0.2
        self.smooth = cfg.label_smoothing

    def forward(self, v: torch.Tensor, t: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:
        B = v.size(0)
        logits = scale * v @ t.T   # (B, B)

        targets = torch.eye(B, device=v.device)
        targets = targets * (1 - self.smooth) + self.smooth / B

        loss_v = -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()
        loss_t = -(targets * F.log_softmax(logits, dim=0)).sum(dim=0).mean()
        loss   = (loss_v + loss_t) / 2

        mask = ~torch.eye(B, dtype=torch.bool, device=v.device)
        neg  = logits[mask].view(B, B-1)
        hard = F.relu(neg - 1/scale + self.margin).mean()
        return loss + 0.1 * hard


# [v2] Thêm hard_neg_margin vào Config nếu chưa có
if not hasattr(cfg, 'hard_neg_margin'):
    cfg.hard_neg_margin = 0.2

print('ContrastiveLoss defined.')

In [ ]:
# CELL 9 — Feature Extraction (Phase 1)
# [v2] Lưu thêm 'mean_t_feat' = mean của tất cả caption embeddings

def extract_and_cache_features(
    records: List[Dict],
    clip_bb: CLIPBackbone,
    cfg,
    split: str
) -> Dict:
    feat_path = Path(cfg.feat_dir) / f'{split}_feats.pt'
    if feat_path.exists():
        logger.info(f'Loading cached {split} features from {feat_path}')
        return torch.load(feat_path, map_location='cpu')

    logger.info(f'Extracting {split} features for {len(records)} videos ...')
    clip_bb.eval().to(cfg.device)
    all_feats = {}

    with torch.no_grad():
        for rec in tqdm(records, desc=f'Extract [{split}]'):
            vid_id   = rec['video_id']
            captions = rec['captions']

            frames = sample_frames(rec['video'], cfg.num_frames)
            if not frames:
                logger.warning(f'No frames for {vid_id}')
                continue
            frames_t = frames_to_tensor(frames, cfg.device)
            v_feat   = clip_bb.encode_frames(frames_t).cpu()

            batch_size = cfg.extract_batch_size
            t_feats = []
            for i in range(0, len(captions), batch_size):
                batch_caps = captions[i:i+batch_size]
                t_feats.append(clip_bb.encode_text(batch_caps).cpu())
            t_feats = torch.cat(t_feats, dim=0)   # (n_caps, proj_dim)

            # [v2] Lưu mean caption embedding
            mean_t_feat = F.normalize(t_feats.mean(dim=0), dim=-1)

            all_feats[vid_id] = {
                'v_feat'    : v_feat,
                'captions'  : captions,
                't_feats'   : t_feats,
                'mean_t_feat': mean_t_feat   # (proj_dim,) — dùng trong v2t eval
            }

    torch.save(all_feats, feat_path)
    logger.info(f'Saved {split} features → {feat_path}  ({len(all_feats)} videos)')
    return all_feats


clip_bb = CLIPBackbone(cfg)
clip_bb = clip_bb.to(cfg.device)

train_feats = extract_and_cache_features(train_data, clip_bb, cfg, 'train')
val_feats   = extract_and_cache_features(val_data,   clip_bb, cfg, 'val')
test_feats  = extract_and_cache_features(test_data,  clip_bb, cfg, 'test')

print(f'Train: {len(train_feats)} | Val: {len(val_feats)} | Test: {len(test_feats)} videos')

In [ ]:
# CELL 10 — Feature Dataset & DataLoader

class FeatureDataset(Dataset):
    """
    Mỗi item: (v_feat, t_feat_random) — caption ngẫu nhiên per step.
    """
    def __init__(self, feats_dict: Dict):
        self.records = []
        for vid_id, val in feats_dict.items():
            self.records.append({
                'v_feat' : val['v_feat'],
                't_feats': val['t_feats'],
            })
        logger.info(f'FeatureDataset: {len(self.records)} videos')

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        v   = rec['v_feat']
        t_all = rec['t_feats']
        cap_idx = random.randint(0, t_all.size(0) - 1)
        return v, t_all[cap_idx]


# Data leak check
train_ids = set(train_feats.keys())
val_ids   = set(val_feats.keys())
test_ids  = set(test_feats.keys())
if train_ids & val_ids:
    logger.warning(f'DATA LEAK: train ∩ val = {len(train_ids & val_ids)} videos!')
if train_ids & test_ids:
    logger.warning(f'DATA LEAK: train ∩ test = {len(train_ids & test_ids)} videos!')
else:
    logger.info('OK: No data leak between splits.')

train_feat_ds = FeatureDataset(train_feats)
train_loader  = DataLoader(
    train_feat_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=True,
    drop_last=True
)
print(f'Train loader: {len(train_loader)} batches x {cfg.batch_size} videos/batch')

In [ ]:
# CELL 11 — Evaluation (v2) — FIXED train/eval consistency + MedR
#
# [v2 fixes]:
# 1. Dùng head.encode_video() / head.encode_text() thay vì bypass branch
# 2. Thêm MedR (Median Rank) cho cả t2v và v2t
# 3. v2t: dùng mean_t_feat (mean caption embedding) thay vì all captions
#    → nhất quán với standard 1K-A test protocol của MSR-VTT
# 4. Bỏ rsum — chỉ trả về R@1, R@5, R@10, MedR

@torch.no_grad()
def evaluate(head: DualEncoderHead, feats_dict: Dict, cfg, desc='Eval') -> Dict:
    """
    Standard MSR-VTT retrieval evaluation.
    - t2v: query=caption, gallery=tất cả videos
    - v2t: query=video, gallery=mean caption của mỗi video
    """
    head.eval()
    device = cfg.device
    vid_ids = list(feats_dict.keys())
    N = len(vid_ids)
    vid2idx = {v: i for i, v in enumerate(vid_ids)}

    # ── Build video pool: encode tất cả video ───────────────────────────
    V_pool = []
    for vid_id in vid_ids:
        v = feats_dict[vid_id]['v_feat'].unsqueeze(0).to(device)  # (1, D)
        V_pool.append(head.encode_video(v).squeeze(0).cpu())
    V_pool = torch.stack(V_pool, dim=0).to(device)   # (N, D)

    # ── Build caption pool (t2v) ─────────────────────────────────────────
    # Mỗi caption là 1 query riêng, GT = video tương ứng
    cap_vecs, cap_gt = [], []
    for vid_id in vid_ids:
        t_feats = feats_dict[vid_id]['t_feats'].to(device)  # (n_caps, D)
        gt_idx  = vid2idx[vid_id]
        for i in range(t_feats.size(0)):
            enc = head.encode_text(t_feats[i:i+1]).squeeze(0).cpu()
            cap_vecs.append(enc)
            cap_gt.append(gt_idx)
    T_all = torch.stack(cap_vecs, dim=0).to(device)    # (N_cap, D)
    gt_vid = torch.tensor(cap_gt, device=device)        # (N_cap,)

    # ── Build mean-caption pool (v2t) ────────────────────────────────────
    # Dùng mean_t_feat đã lưu sẵn
    T_mean = []
    for vid_id in vid_ids:
        mt = feats_dict[vid_id]['mean_t_feat'].unsqueeze(0).to(device)  # (1, D)
        T_mean.append(head.encode_text(mt).squeeze(0).cpu())
    T_mean = torch.stack(T_mean, dim=0).to(device)   # (N, D)

    # ── t2v: sim (N_cap, N_vid) ──────────────────────────────────────────
    sim_t2v = T_all @ V_pool.T

    def recall_medr(sim, gt_idx_per_query):
        """Tính R@1, R@5, R@10, MedR từ similarity matrix."""
        N_q = sim.size(0)
        ranks = []
        for i in range(N_q):
            scores = sim[i]                                    # (N_gallery,)
            sorted_idx = scores.argsort(descending=True)      # (N_gallery,)
            gt = gt_idx_per_query[i].item()
            rank = (sorted_idx == gt).nonzero(as_tuple=True)[0].item() + 1  # 1-indexed
            ranks.append(rank)
        ranks = np.array(ranks)
        return {
            'R@1' : (ranks <= 1).mean() * 100,
            'R@5' : (ranks <= 5).mean() * 100,
            'R@10': (ranks <= 10).mean() * 100,
            'MedR': float(np.median(ranks)),
        }

    t2v = recall_medr(sim_t2v, gt_vid)

    # ── v2t: sim (N_vid, N_vid) — video vs mean-caption ─────────────────
    sim_v2t = V_pool @ T_mean.T
    gt_vid_v2t = torch.arange(N, device=device)  # GT: video i → caption i
    v2t = recall_medr(sim_v2t, gt_vid_v2t)

    metrics = {
        't2v_R@1' : t2v['R@1'],  't2v_R@5' : t2v['R@5'],
        't2v_R@10': t2v['R@10'], 't2v_MedR': t2v['MedR'],
        'v2t_R@1' : v2t['R@1'],  'v2t_R@5' : v2t['R@5'],
        'v2t_R@10': v2t['R@10'], 'v2t_MedR': v2t['MedR'],
    }

    logger.info(
        f'[{desc}] '
        f't2v R@1={metrics["t2v_R@1"]:.2f} R@5={metrics["t2v_R@5"]:.2f} '
        f'R@10={metrics["t2v_R@10"]:.2f} MedR={metrics["t2v_MedR"]:.1f} | '
        f'v2t R@1={metrics["v2t_R@1"]:.2f} R@5={metrics["v2t_R@5"]:.2f} '
        f'R@10={metrics["v2t_R@10"]:.2f} MedR={metrics["v2t_MedR"]:.1f}'
    )
    return metrics


print('evaluate() v2 defined. [MedR added, KAN bypass removed]')

In [ ]:
# CELL 12 — Trainer (v2)
# [v2] Early stopping dùng t2v_R@1 thay vì Rsum

class Trainer:
    def __init__(self, head: DualEncoderHead, train_loader, val_feats, cfg):
        self.head         = head.to(cfg.device)
        self.train_loader = train_loader
        self.val_feats    = val_feats
        self.cfg          = cfg
        self.criterion    = ContrastiveLoss(cfg)
        self.scaler       = GradScaler(enabled=cfg.use_amp)

        self.optimizer = torch.optim.AdamW(
            self.head.parameters(),
            lr=cfg.lr_head, weight_decay=cfg.weight_decay
        )
        total_steps  = len(train_loader) * cfg.num_epochs // cfg.accum_steps
        warmup_steps = int(total_steps * cfg.warmup_ratio)
        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer, warmup_steps, total_steps
        )

        self.history = {
            'train_loss': [],
            't2v_R1': [], 't2v_R5': [], 't2v_R10': [], 't2v_MedR': [],
            'v2t_R1': [], 'v2t_R5': [], 'v2t_R10': [], 'v2t_MedR': [],
        }
        # [v2] Early stopping theo t2v_R@1 (bỏ Rsum)
        self.best_r1    = 0.0
        self.bad_epochs = 0

    def train_epoch(self, epoch: int) -> float:
        self.head.train()
        total_loss = 0.0
        self.optimizer.zero_grad()

        for step, (v_feat, t_feat) in enumerate(self.train_loader):
            v_feat = v_feat.to(self.cfg.device)
            t_feat = t_feat.to(self.cfg.device)

            with autocast(device_type='cuda', enabled=self.cfg.use_amp):
                v_out, t_out, scale = self.head(v_feat, t_feat)
                loss = self.criterion(v_out, t_out, scale)
                loss = loss / self.cfg.accum_steps

            self.scaler.scale(loss).backward()

            if (step + 1) % self.cfg.accum_steps == 0:
                self.scaler.unscale_(self.optimizer)
                nn.utils.clip_grad_norm_(self.head.parameters(), self.cfg.grad_clip)
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.scheduler.step()
                self.optimizer.zero_grad()
                with torch.no_grad():
                    self.head.logit_scale.clamp_(0.0, self.cfg.logit_scale_max)

            total_loss += loss.item() * self.cfg.accum_steps

            if (step + 1) % self.cfg.log_every == 0:
                lr = self.optimizer.param_groups[0]['lr']
                logger.info(
                    f'Ep {epoch} step {step+1}/{len(self.train_loader)} '
                    f'loss={total_loss/(step+1):.4f} lr={lr:.2e}'
                )

        return total_loss / len(self.train_loader)

    def run(self):
        for epoch in range(1, self.cfg.num_epochs + 1):
            t0 = time.time()
            avg_loss = self.train_epoch(epoch)
            self.history['train_loss'].append(avg_loss)

            if epoch % self.cfg.eval_every == 0:
                metrics = evaluate(self.head, self.val_feats, self.cfg, desc=f'Val Ep{epoch}')

                # Log history
                self.history['t2v_R1'].append(metrics['t2v_R@1'])
                self.history['t2v_R5'].append(metrics['t2v_R@5'])
                self.history['t2v_R10'].append(metrics['t2v_R@10'])
                self.history['t2v_MedR'].append(metrics['t2v_MedR'])
                self.history['v2t_R1'].append(metrics['v2t_R@1'])
                self.history['v2t_R5'].append(metrics['v2t_R@5'])
                self.history['v2t_R10'].append(metrics['v2t_R@10'])
                self.history['v2t_MedR'].append(metrics['v2t_MedR'])

                r1 = metrics['t2v_R@1']   # [v2] dùng t2v_R@1 làm pivot

                logger.info(
                    f'Epoch {epoch:3d} | loss={avg_loss:.4f} '
                    f'| t2v R@1={metrics["t2v_R@1"]:.2f} R@5={metrics["t2v_R@5"]:.2f} '
                    f'R@10={metrics["t2v_R@10"]:.2f} MedR={metrics["t2v_MedR"]:.1f} '
                    f'| v2t R@1={metrics["v2t_R@1"]:.2f} '
                    f'| {time.time()-t0:.1f}s'
                )

                if r1 > self.best_r1 + self.cfg.min_delta:
                    self.best_r1 = r1
                    self.bad_epochs = 0
                    torch.save(
                        self.head.state_dict(),
                        Path(self.cfg.ckpt_dir) / 'best_head.pt'
                    )
                    logger.info(f'  ** New best t2v R@1={r1:.2f} — saved.')
                else:
                    self.bad_epochs += 1
                    if self.bad_epochs >= self.cfg.patience:
                        logger.info(f'Early stopping at epoch {epoch} (no improvement for {self.cfg.patience} epochs).')
                        break

        return self.history


print('Trainer v2 defined. [early stopping on t2v_R@1]')

In [ ]:
# CELL 13 — Phase 2: Train Head

head    = DualEncoderHead(cfg)
trainer = Trainer(head, train_loader, val_feats, cfg)
history = trainer.run()

print('\nTraining complete.')
print(f'Best val t2v R@1: {trainer.best_r1:.2f}')

In [ ]:
# CELL 14 — Plot Training Curves
import matplotlib.pyplot as plt

epochs = list(range(1, len(history['train_loss']) + 1))
eval_epochs = list(range(1, len(history['t2v_R1']) + 1))

fig, axes = plt.subplots(2, 3, figsize=(18, 8))

axes[0,0].plot(epochs, history['train_loss'])
axes[0,0].set_title('Train Loss'); axes[0,0].set_xlabel('Epoch')

axes[0,1].plot(eval_epochs, history['t2v_R1'], label='t2v')
axes[0,1].plot(eval_epochs, history['v2t_R1'], label='v2t')
axes[0,1].legend(); axes[0,1].set_title('R@1'); axes[0,1].set_xlabel('Epoch')

axes[0,2].plot(eval_epochs, history['t2v_R5'], label='t2v')
axes[0,2].plot(eval_epochs, history['v2t_R5'], label='v2t')
axes[0,2].legend(); axes[0,2].set_title('R@5'); axes[0,2].set_xlabel('Epoch')

axes[1,0].plot(eval_epochs, history['t2v_R10'], label='t2v')
axes[1,0].plot(eval_epochs, history['v2t_R10'], label='v2t')
axes[1,0].legend(); axes[1,0].set_title('R@10'); axes[1,0].set_xlabel('Epoch')

axes[1,1].plot(eval_epochs, history['t2v_MedR'], label='t2v', color='red')
axes[1,1].plot(eval_epochs, history['v2t_MedR'], label='v2t', color='orange')
axes[1,1].legend(); axes[1,1].set_title('MedR (lower=better)'); axes[1,1].set_xlabel('Epoch')
axes[1,1].invert_yaxis()

axes[1,2].axis('off')  # placeholder

plt.tight_layout()
curves_path = Path(cfg.ckpt_dir) / 'training_curves_v2.png'
plt.savefig(curves_path, dpi=120)
plt.show()
print(f'Saved → {curves_path}')

In [ ]:
# CELL 15 — Test Set Evaluation (Final)

best_head = DualEncoderHead(cfg)
best_head.load_state_dict(
    torch.load(Path(cfg.ckpt_dir) / 'best_head.pt', map_location=cfg.device)
)
best_head = best_head.to(cfg.device)

test_metrics = evaluate(best_head, test_feats, cfg, desc='TEST')

print('\n' + '='*50)
print('       MSR-VTT Test Results (v2)')
print('='*50)
print(f'           R@1    R@5   R@10   MedR')
print(f'Text→Video {test_metrics["t2v_R@1"]:6.2f} {test_metrics["t2v_R@5"]:6.2f} {test_metrics["t2v_R@10"]:6.2f} {test_metrics["t2v_MedR"]:6.1f}')
print(f'Video→Text {test_metrics["v2t_R@1"]:6.2f} {test_metrics["v2t_R@5"]:6.2f} {test_metrics["v2t_R@10"]:6.2f} {test_metrics["v2t_MedR"]:6.1f}')
print('='*50)

In [ ]:
# CELL 16 — (Optional) Phase 3: Fine-tune CLIP last 2 layers
# [v2] Tương thích với DualEncoderHead

FINETUNE_CLIP = False   # đặt True để bật

if FINETUNE_CLIP:
    logger.info('Phase 3: Fine-tuning CLIP last 2 vision layers + proj...')
    clip_bb.unfreeze_last_n_layers(n=2)

    # Optimizer riêng với LR nhỏ cho CLIP, LR vừa cho head
    optimizer_ft = torch.optim.AdamW([
        {'params': clip_bb.vision_proj.parameters(), 'lr': cfg.lr_proj},
        {'params': clip_bb.text_proj.parameters(),   'lr': cfg.lr_proj},
        {'params': [p for p in clip_bb.model.parameters() if p.requires_grad],
         'lr': cfg.lr_proj * 0.1},
        {'params': best_head.parameters(), 'lr': cfg.lr_head * 0.3},
    ], weight_decay=cfg.weight_decay)

    criterion = ContrastiveLoss(cfg)
    clip_bb.train()
    best_head.train()

    FT_EPOCHS = 5
    SUBSET = 500  # video để demo; tăng lên len(train_data) để train full

    for ep in range(1, FT_EPOCHS + 1):
        ep_loss = 0.0
        for rec in tqdm(train_data[:SUBSET], desc=f'FT Ep {ep}'):
            frames = sample_frames(rec['video'], cfg.num_frames)
            if not frames: continue
            frames_t = frames_to_tensor(frames, cfg.device)
            v_feat = clip_bb.encode_frames(frames_t).unsqueeze(0)

            cap = random.choice(rec['captions'])
            t_feat = clip_bb.encode_text([cap])

            v_out, t_out, scale = best_head(v_feat, t_feat)
            loss = criterion(v_out, t_out, scale)
            loss.backward()
            optimizer_ft.step(); optimizer_ft.zero_grad()
            ep_loss += loss.item()
        logger.info(f'FT Ep {ep} loss={ep_loss/SUBSET:.4f}')

    # Re-extract và evaluate
    val_feats_ft = extract_and_cache_features(val_data, clip_bb, cfg, 'val_ft')
    ft_metrics   = evaluate(best_head, val_feats_ft, cfg, desc='FT Val')
    print(f'FT Val t2v R@1={ft_metrics["t2v_R@1"]:.2f}  MedR={ft_metrics["t2v_MedR"]:.1f}')
else:
    print('Phase 3 skipped (FINETUNE_CLIP=False).')